## Severity Analysis Overview

This script performs mental health analysis using machine learning. It starts by loading a dataset of Reddit posts (`mergeddata.csv`) and cleaning the `Selftext` column by lowercasing, removing punctuation, tokenizing, and removing stopwords. It uses keyword matching to simulate disorder labels (Anxiety, Depression, ADHD) and assigns a severity score based on the presence of intensifiers, qualifiers, and medication mentions.

The processed text is then vectorized using TF-IDF to extract features, and a Random Forest Classifier is trained to predict mental health disorders, while a Random Forest Regressor estimates severity levels. The model's performance is evaluated using classification metrics and regression accuracy. Finally, the script applies the models to sample inputs, generates predictions for the entire dataset, and saves the enhanced results (with predicted disorder and severity) to `mergeddata_with_ml_predictions.csv`.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/mergeddata.csv')

In [ ]:
df.head()

,Author,Created_UTC,Created_DateTime,Score,Selftext,Tokenized_Selftext,Subreddit,Title
0,Flashy_Highway_8937,1.739054e+09,2025-02-08 22:33:09,2,Hi.\n\nLike many I suffer from bad health anxi...,"['Hi', '.', 'Like', 'many', 'I', 'suffer', 'fr...",Anxiety,Chest Strain or heart issue
1,Timdillonfanclub,1.739054e+09,2025-02-08 22:29:42,1,I was prescribed propranolol for my anxiety. ...,"['I', 'was', 'prescribed', 'propranolol', 'for...",Anxiety,Trying propranolol
2,user12747,1.739053e+09,2025-02-08 22:24:55,1,Our house is 1970s and it’s very likely it has...,"['Our', 'house', 'is', '1970s', 'and', 'it', '...",Anxiety,Super worried about asbestos exposure
3,Nique_T,1.739053e+09,2025-02-08 22:22:57,2,Is anyone waking up feeling weak or drained? I...,"['Is', 'anyone', 'waking', 'up', 'feeling', 'w...",Anxiety,Is anyone
4,scubasteve_XYZ,1.739053e+09,2025-02-08 22:22:16,1,I am going nuts.\nMy boyfriend and I broke up ...,"['I', 'am', 'going', 'nuts', '.', 'My', 'boyfr...",Anxiety,Sometimes I’m okay and then shanties I remembe...


## Severity Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error, r2_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re


nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


df = pd.read_csv('mergeddata.csv')
df['Selftext'] = df['Selftext'].fillna('')


def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['Processed_Selftext'] = df['Selftext'].apply(preprocess_text)

anxiety_keywords = {
    "chest pain", "chest tightness", "lightheadedness", "nausea", "often", "racing thoughts",
    "can't stop", "scared", "insomnia", "anxiety attack", "headaches", "can't stop worrying",
    "shaking", "can't sleep", "bathroom issues", "diarrhea", "dizzy", "health anxiety",
    "vomit", "throwing up", "panic attack", "everyday", "extreme", "terrified", "heart racing",
    "benzos", "nervous breakdown", "agoraphobia"
}

depression_keywords = {
    "sad", "down", "lonely", "die", "dying", "kill myself", "hopeless", "worthless",
    "alone", "pain", "can’t take this", "giving up", "exhausted", "crying", "numb",
    "losing weight", "gaining weight"
}

adhd_keywords = {
    "impulsive", "impulsivity", "keep track of", "losing things", "medication",
    "stimulants", "motivation", "procrastination", "productivity", "focus", "planning",
    "task paralysis", "forgetful", "restlessness", "distracted", "disorganized",
    "fidgeting", "interrupting"
}

def simulate_training_disorder(selftext, subreddit):
    text_lower = str(selftext).lower()
    depression_score = sum(1 for word in depression_keywords if word in text_lower)
    adhd_score = sum(1 for word in adhd_keywords if word in text_lower)
    anxiety_score = sum(1 for word in anxiety_keywords if word in text_lower)

    if depression_score > 0:
        return "Depression"
    elif adhd_score > 0 and subreddit == "ADHD":
        return "ADHD"
    elif anxiety_score > 0 and subreddit == "Anxiety":
        return "Anxiety"
    else:
        return subreddit if subreddit in ["ADHD", "Anxiety"] else "Depression"


def simulate_severity(selftext, disorder):
    intensifiers = ["severe", "constant", "every day", "terrified", "cant stop", "extreme", "everyday"]
    qualifiers = ["mild", "sometimes", "occasional"]
    medications = ["xanax", "prozac", "zoloft", "vyvanse", "adderall", "ssri", "lexapro",
                   "concerta", "ritalin", "propranolol", "benzos", "stimulants", "medication"]

    text_lower = str(selftext).lower()
    if disorder == "Anxiety":
        min_score, max_score = 5, 10
        keywords = anxiety_keywords
    elif disorder == "Depression":
        min_score, max_score = 4, 10
        keywords = depression_keywords
    elif disorder == "ADHD":
        min_score, max_score = 4, 10
        keywords = adhd_keywords
    else:
        raise ValueError("Disorder must be Anxiety, Depression, or ADHD")

    score = 1
    keyword_count = sum(1 for word in keywords if word in text_lower)
    if keyword_count > 0:
        score = min_score
        if keyword_count >= 3:
            score = min_score + 2

    for intensifier in intensifiers:
        if intensifier in text_lower:
            score = min(score + 1, max_score)
            break
    for qualifier in qualifiers:
        if qualifier in text_lower:
            score = max(score - 1, min_score)
            break

    if any(med in text_lower for med in medications):
        score = max(score, min_score + 1)

    return max(min_score, min(score, max_score))


df['Training_Disorder'] = df.apply(
    lambda row: simulate_training_disorder(row['Selftext'], row['Subreddit']), axis=1
)
df['Simulated_Severity'] = df.apply(
    lambda row: simulate_severity(row['Selftext'], row['Training_Disorder']), axis=1
)


vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 3))
X_disorder = vectorizer.fit_transform(df['Processed_Selftext'])
y_disorder = df['Training_Disorder']


X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_disorder, y_disorder, test_size=0.2, random_state=42)


clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_d, y_train_d)


y_pred_d = clf.predict(X_test_d)
print("Disorder Prediction Evaluation:")
print(classification_report(y_test_d, y_pred_d))
disorder_accuracy = np.mean(y_pred_d == y_test_d)
print(f"Disorder Prediction Accuracy: {disorder_accuracy:.2%}")


X_severity = X_disorder
y_severity = df['Simulated_Severity']


X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_severity, y_severity, test_size=0.2, random_state=42)


regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train_s, y_train_s)


y_pred_s = regressor.predict(X_test_s)
mse = mean_squared_error(y_test_s, y_pred_s)
r2 = r2_score(y_test_s, y_pred_s)
severity_accuracy = np.mean(np.abs(y_test_s - y_pred_s) <= 1)
print(f"Severity Prediction Mean Squared Error: {mse:.2f}")
print(f"Severity Prediction R-squared: {r2:.2f}")
print(f"Severity Prediction Pseudo-Accuracy (within ±1): {severity_accuracy:.2%}")


def predict_disorder_and_severity(selftext):
    processed_text = preprocess_text(selftext)
    X_new = vectorizer.transform([processed_text])


    disorder = clf.predict(X_new)[0]
    severity = regressor.predict(X_new)[0]

    if disorder == "Anxiety":
        severity = max(5, min(10, severity))
    elif disorder in ["Depression", "ADHD"]:
        severity = max(4, min(10, severity))

    return disorder, round(severity)


test_indices = [0, 1, 2, 10, 20]
for idx in test_indices:
    selftext = df['Selftext'].iloc[idx]
    subreddit = df['Subreddit'].iloc[idx]
    disorder, severity = predict_disorder_and_severity(selftext)
    print(f"Selftext: {selftext[:50]}...")
    print(f"Actual Subreddit: {subreddit}")
    print(f"Predicted Disorder: {disorder}")
    print(f"Predicted Severity: {severity}\n")

df[['Predicted_Disorder', 'Predicted_Severity']] = df['Selftext'].apply(
    lambda x: pd.Series(predict_disorder_and_severity(x))
)
df.to_csv('mergeddata_with_ml_predictions.csv', index=False)
print("Predictions saved to 'mergeddata_with_ml_predictions.csv'")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Disorder Prediction Evaluation:
              precision    recall  f1-score   support

        ADHD       0.80      0.58      0.68       505
     Anxiety       0.75      0.41      0.53       474
  Depression       0.73      0.91      0.81      1382

    accuracy                           0.74      2361
   macro avg       0.76      0.64      0.67      2361
weighted avg       0.75      0.74      0.72      2361

Disorder Prediction Accuracy: 73.99%
Severity Prediction Mean Squared Error: 0.33
Severity Prediction R-squared: 0.43
Severity Prediction Pseudo-Accuracy (within ±1): 93.18%
Selftext: Hi.

Like many I suffer from bad health anxiety. A...
Actual Subreddit: Anxiety
Predicted Disorder: Depression
Predicted Severity: 5

Selftext: I was prescribed propranolol for my anxiety.  I do...
Actual Subreddit: Anxiety
Predicted Disorder: Anxiety
Predicted Severity: 6

Selftext: Our house is 1970s and it’s very likely it has asb...
Actual Subreddit: Anxiety
Predicted Disorder: Anxiety
Predicted 

In [ ]:
df_new = pd.read_csv('/content/mergeddata_with_ml_predictions.csv')

In [ ]:
pd.reset_option('display.max_colwidth')

In [ ]:
df_new[['Selftext', 'Subreddit','Training_Disorder', 'Predicted_Disorder'
,'Simulated_Severity', 'Predicted_Severity']].head(5)


,Selftext,Subreddit,Training_Disorder,Predicted_Disorder,Simulated_Severity,Predicted_Severity
0,Hi.\n\nLike many I suffer from bad health anxi...,Anxiety,Depression,Depression,4,5
1,I was prescribed propranolol for my anxiety. ...,Anxiety,Anxiety,Anxiety,6,6
2,Our house is 1970s and it’s very likely it has...,Anxiety,Anxiety,Anxiety,5,5
3,Is anyone waking up feeling weak or drained? I...,Anxiety,Anxiety,Anxiety,5,5
4,I am going nuts.\nMy boyfriend and I broke up ...,Anxiety,Depression,Depression,4,4


In [ ]:
df_new = df_new.drop(columns=['Training_Disorder', 'Simulated_Severity'])

In [ ]:
df_new = pd.read_csv('/content/mergeddata_with_ml_predictions.csv')
df_new[['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

,Selftext,Subreddit,Predicted_Disorder,Predicted_Severity
0,Hi.\n\nLike many I suffer from bad health anxi...,Anxiety,Depression,5
1,I was prescribed propranolol for my anxiety. ...,Anxiety,Anxiety,6
2,Our house is 1970s and it’s very likely it has...,Anxiety,Anxiety,5
3,Is anyone waking up feeling weak or drained? I...,Anxiety,Anxiety,5
4,I am going nuts.\nMy boyfriend and I broke up ...,Anxiety,Depression,4


In [ ]:
df_new[df_new['Subreddit'] == 'depression'][['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

,Selftext,Subreddit,Predicted_Disorder,Predicted_Severity
987,There is nothing I truly like. I don't want to...,depression,Depression,4
988,"Hi guys, I've (35/m) been holding this inside ...",depression,Depression,5
989,"I am just so down and tired of everything, i ...",depression,Depression,4
990,Due to my job I cannot go to a doctor or go to...,depression,Depression,4
991,I've dealt with depression for as long as I ca...,depression,Depression,5


In [ ]:
df_new[df_new['Subreddit'] == 'ADHD'][['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

,Selftext,Subreddit,Predicted_Disorder,Predicted_Severity
1974,I know there’s a few posts regarding this but ...,ADHD,ADHD,5
1975,I was wondering if I’m the only one who feels ...,ADHD,Depression,4
1976,So I'm (30M) and I have been dealing with thi...,ADHD,ADHD,5
1977,"Hey fellow cooks,\n\nI love cooking, but I hat...",ADHD,ADHD,4
1978,what happens to you when you drink? either dru...,ADHD,ADHD,4


In [ ]:
df_new[['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

,Selftext,Subreddit,Predicted_Disorder,Predicted_Severity
0,Hi.\n\nLike many I suffer from bad health anxi...,Anxiety,Depression,5
1,I was prescribed propranolol for my anxiety. ...,Anxiety,Anxiety,6
2,Our house is 1970s and it’s very likely it has...,Anxiety,Anxiety,5
3,Is anyone waking up feeling weak or drained? I...,Anxiety,Anxiety,5
4,I am going nuts.\nMy boyfriend and I broke up ...,Anxiety,Depression,4


In [ ]:
df_new[['Selftext', 'Subreddit', 'Predicted_Disorder', 'Predicted_Severity']].head(5)

,Selftext,Subreddit,Predicted_Disorder,Predicted_Severity
0,Hi.\n\nLike many I suffer from bad health anxi...,Anxiety,Depression,5
1,I was prescribed propranolol for my anxiety. ...,Anxiety,Anxiety,6
2,Our house is 1970s and it’s very likely it has...,Anxiety,Anxiety,5
3,Is anyone waking up feeling weak or drained? I...,Anxiety,Anxiety,5
4,I am going nuts.\nMy boyfriend and I broke up ...,Anxiety,Depression,4


In [ ]:
df_new[df_new['Subreddit'] == 'ADHD'].head(10)

,Author,Created_UTC,Created_DateTime,Score,Selftext,Tokenized_Selftext,Subreddit,Title,Processed_Selftext,Training_Disorder,Simulated_Severity,Predicted_Disorder,Predicted_Severity
1974,girlsledisko,1.739054e+09,2025-02-08 22:33:30,1,I know there’s a few posts regarding this but ...,"['I', 'know', 'there', '’', 's', 'a', 'few', '...",ADHD,Body-focused repetitive behaviors (skin pickin...,know theres posts regarding id love fresh disc...,ADHD,5,ADHD,5
1975,Serious-Chipmunk-872,1.739054e+09,2025-02-08 22:25:25,2,I was wondering if I’m the only one who feels ...,"['I', 'was', 'wondering', 'if', 'I', '’', 'm',...",ADHD,I feel awkward around people,wondering im one feels really uncomfortable pe...,Depression,4,Depression,4
1976,canviskillr,1.739053e+09,2025-02-08 22:18:22,6,So I'm (30M) and I have been dealing with thi...,"['So', 'I', ""'m"", '(', '30M', ')', 'and', 'I',...",ADHD,My inability to delay gratification is ruining...,im 30m dealing things like binge eating relati...,ADHD,5,ADHD,5
1977,ddebie,1.739053e+09,2025-02-08 22:16:49,2,"Hey fellow cooks,\n\nI love cooking, but I hat...","['Hey', 'fellow', 'cooks', ',', 'I', 'love', '...",ADHD,Cooking with ADHD? This App Turns Recipes into...,hey fellow cooks love cooking hate scrolling e...,ADHD,4,ADHD,4
1978,hawaiiwater2,1.739053e+09,2025-02-08 22:14:04,1,what happens to you when you drink? either dru...,"['what', 'happens', 'to', 'you', 'when', 'you'...",ADHD,What happens when you drink?,happens drink either drunk little alcohol depr...,ADHD,4,ADHD,4
1979,nobananabread,1.739053e+09,2025-02-08 22:13:53,1,Hello!\n\nI have a few questions and to prefac...,"['Hello', '!', 'I', 'have', 'a', 'few', 'quest...",ADHD,Getting a diagnosis after taking adderall,hello questions preface im international stude...,Depression,5,Depression,5
1980,Caitlyn3030,1.739053e+09,2025-02-08 22:10:16,2,I recently got an ADHD diagnosis as an adult a...,"['I', 'recently', 'got', 'an', 'ADHD', 'diagno...",ADHD,Coffee + Vyvanse?,recently got adhd diagnosis adult trialling vy...,ADHD,5,ADHD,5
1981,vincevsound,1.739052e+09,2025-02-08 22:03:32,1,Hi all my son takes Ritalin IR. He can’t take ...,"['Hi', 'all', 'my', 'son', 'takes', 'Ritalin',...",ADHD,Different doses of Ritalin throughout the day?,hi son takes ritalin ir cant take la gets supe...,ADHD,5,ADHD,5
1982,Danaheartssssss,1.739051e+09,2025-02-08 21:49:45,2,"I have trouble remembering appointments, so I ...","['I', 'have', 'trouble', 'remembering', 'appoi...",ADHD,How Do You Keep Track of Appointments?,trouble remembering appointments usually write...,Depression,4,Depression,4
1983,Low_Buy_1421,1.739050e+09,2025-02-08 21:24:55,2,I know this is a very common side effect from ...,"['I', 'know', 'this', 'is', 'a', 'very', 'comm...",ADHD,No appetite from medication :(,know common side effect adhd medications dont ...,ADHD,5,ADHD,5


In [ ]:
df_new[df_new['Subreddit'] == 'depression'].head(10)

,Author,Created_UTC,Created_DateTime,Score,Selftext,Tokenized_Selftext,Subreddit,Title,Processed_Selftext,Training_Disorder,Simulated_Severity,Predicted_Disorder,Predicted_Severity
987,IHTSFR,1.739054e+09,2025-02-08 22:28:18,1,There is nothing I truly like. I don't want to...,"['There', 'is', 'nothing', 'I', 'truly', 'like...",depression,I want to die,nothing truly like dont want struggle grow old...,Depression,4,Depression,4
988,colinofrivia,1.739053e+09,2025-02-08 22:21:18,2,"Hi guys, I've (35/m) been holding this inside ...","['Hi', 'guys', ',', 'I', ""'ve"", '(', '35/m', '...",depression,I've suffered with depression my entire life a...,hi guys ive 35m holding inside long actually d...,Depression,5,Depression,5
989,Disastrous-Silver838,1.739053e+09,2025-02-08 22:19:10,1,"I am just so down and tired of everything, i ...","['I', 'am', 'just', 'so', 'down', 'and', 'tire...",depression,"I am sad, down and just alone",tired everything spent last 9 months fighting ...,Depression,4,Depression,4
990,Sure-Ad314,1.739053e+09,2025-02-08 22:19:04,1,Due to my job I cannot go to a doctor or go to...,"['Due', 'to', 'my', 'job', 'I', 'can', 'not', ...",depression,Over the Counter Anti-Depressants or Anxiety M...,due job go doctor go therapy also take anythin...,Depression,4,Depression,4
991,godddamnitsarah,1.739053e+09,2025-02-08 22:09:11,2,I've dealt with depression for as long as I ca...,"['I', ""'ve"", 'dealt', 'with', 'depression', 'f...",depression,It's not getting easier.,ive dealt depression long remember teen gradua...,Depression,5,Depression,5
992,Weak_Strategy_9426,1.739052e+09,2025-02-08 22:03:10,10,21F\n\nIt’s my birthday today and I hate it. M...,"['21F', 'It', '’', 's', 'my', 'birthday', 'tod...",depression,I can’t take this pain anymore,21f birthday today hate negative thoughts get ...,Depression,4,Depression,4
993,DearWorldliness9654,1.739052e+09,2025-02-08 22:01:32,1,I’m so fed up with life idk what to do anymore...,"['I', '’', 'm', 'so', 'fed', 'up', 'with', 'li...",depression,Help,im fed life idk anymore almost 2 years nothing...,Depression,4,Depression,4
994,CryptographerDue4624,1.739052e+09,2025-02-08 21:54:53,2,i’m almost 31. i have no job. no close friends...,"['i', '’', 'm', 'almost', '31.', 'i', 'have', ...",depression,how to find the urge to keep living,im almost 31 job close friends anymore live pa...,Depression,5,Depression,5
995,fauxfoxem,1.739052e+09,2025-02-08 21:54:31,2,I was diagnosed with depression at 14. I was l...,"['I', 'was', 'diagnosed', 'with', 'depression'...",depression,"Technically diagnosed with depression, but hav...",diagnosed depression 14 later diagnosed autism...,Depression,5,Depression,5
996,enchantingsunsetblvd,1.739051e+09,2025-02-08 21:49:41,3,But that place doesn’t exist anymore. and it n...,"['But', 'that', 'place', 'doesn', '’', 't', 'e...",depression,I want to go home,place doesnt exist anymore never,Depression,4,Depression,4


In [ ]:
df_new[(df_new["Predicted_Severity"] > 5) & (df["Subreddit"] == "depression")].head()

,Author,Created_UTC,Created_DateTime,Score,Selftext,Tokenized_Selftext,Subreddit,Title,Processed_Selftext,Predicted_Disorder,Predicted_Severity
1025,AMSTafty,1.739045e+09,2025-02-08 19:57:52,2,"\n\n*** LONG POST***\n\nHello friends, I hope ...","['*', '*', '*', 'LONG', 'POST', '*', '*', '*',...",depression,"I had a break down, and I was scared.",long post hello friends hope good decent day f...,Depression,7
1029,NaN,1.739043e+09,2025-02-08 19:36:17,1,"I'm a lurker in all regards, never liking or d...","['I', ""'m"", 'a', 'lurker', 'in', 'all', 'regar...",depression,a no lifer with no hobbies or interests do peo...,im lurker regards never liking downvoting post...,Depression,6
1036,Pristine_Tell_2450,1.739042e+09,2025-02-08 19:09:48,1,"Its annoying that i understand my problems, i ...","['Its', 'annoying', 'that', 'i', 'understand',...",depression,What do i do? Where do i even start? Im tired ...,annoying understand problems know stem dont kn...,Depression,6
1038,GoalRealistic6020,1.739041e+09,2025-02-08 19:00:48,1,can anyone tell me if this is okay to give to ...,"['can', 'anyone', 'tell', 'me', 'if', 'this', ...",depression,a letter to my teacher,anyone tell okay give teacher keeps asking im ...,Depression,6
1058,FeelingPie6750,1.739034e+09,2025-02-08 17:07:12,1,I always come to the conclusion that I should ...,"['I', 'always', 'come', 'to', 'the', 'conclusi...",depression,Someone remind me what exactly is the point of...,always come conclusion go hang never go closes...,Depression,6
